In [1]:
import io # Input/Output Module
import os # OS interfaces
import cv2 # OpenCV package
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

from urllib import request # module for opening HTTP requests
from matplotlib import pyplot as plt # Plotting library
from sklearn.neural_network import MLPClassifier

In [2]:
# Input data files are available in the read-only "../input/" directory

train = pd.read_csv(
    './kaggle/input/kul-computer-vision-ga-1-2025/train_set.csv', index_col = 0)
train.index = train.index.rename('id')

test = pd.read_csv(
    './kaggle/input/kul-computer-vision-ga-1-2025/test_set.csv', index_col = 0)
test.index = test.index.rename('id')

#Read in the preprocessed images and store in the correct data containers
features = np.load('group16_features.npz', allow_pickle=False) 

#training data, the extracted features + the preprocessed images
#hog_features
hog_train = features["hog_train"]
hog_test = features["hog_test"]
#SIFT features
sift_train = features["sift_train"]
sift_test = features["sift_test"]


labels_train = np.concatenate((features["train_y"],features["train_y"]),axis=0)
images_train = np.load('./kaggle/working/prepped_data/train_X.npy', allow_pickle=False)
images_train2 = np.concatenate((images_train,images_train[:, :, ::-1, :]),axis=0)


images_test = np.load('./kaggle/working/prepped_data/test_X.npy', allow_pickle=False)




#train_size, test_size = len(train),len(test)

#"The training set contains {} examples, the test set contains {} examples.".format(train_size, test_size)

## Simple forward thing

In [3]:
clf = MLPClassifier(solver='adam', 
                    activation='relu', 
                    alpha=.001, 
                    hidden_layer_sizes=(512, 128, 3), 
                    random_state=1, 
                    max_iter=50)
clf.fit(hog_train, labels_train)

,"hidden_layer_sizes hidden_layer_sizes: array-like of shape(n_layers - 2,), default=(100,)The ith element represents the number of neurons in the ithhidden layer.","(512, ...)"
,"activation activation: {'identity', 'logistic', 'tanh', 'relu'}, default='relu'Activation function for the hidden layer.- 'identity', no-op activation, useful to implement linear bottleneck, returns f(x) = x- 'logistic', the logistic sigmoid function, returns f(x) = 1 / (1 + exp(-x)).- 'tanh', the hyperbolic tan function, returns f(x) = tanh(x).- 'relu', the rectified linear unit function, returns f(x) = max(0, x)",'relu'
,"solver solver: {'lbfgs', 'sgd', 'adam'}, default='adam'The solver for weight optimization.- 'lbfgs' is an optimizer in the family of quasi-Newton methods.- 'sgd' refers to stochastic gradient descent.- 'adam' refers to a stochastic gradient-based optimizer proposed by Kingma, Diederik, and Jimmy BaFor a comparison between Adam optimizer and SGD, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_training_curves.py`.Note: The default solver 'adam' works pretty well on relativelylarge datasets (with thousands of training samples or more) in terms ofboth training time and validation score.For small datasets, however, 'lbfgs' can converge faster and performbetter.",'adam'
,"alpha alpha: float, default=0.0001Strength of the L2 regularization term. The L2 regularization termis divided by the sample size when added to the loss.For an example usage and visualization of varying regularization, see:ref:`sphx_glr_auto_examples_neural_networks_plot_mlp_alpha.py`.",0.001
,"batch_size batch_size: int, default='auto'Size of minibatches for stochastic optimizers.If the solver is 'lbfgs', the classifier will not use minibatch.When set to ""auto"", `batch_size=min(200, n_samples)`.",'auto'
,"learning_rate learning_rate: {'constant', 'invscaling', 'adaptive'}, default='constant'Learning rate schedule for weight updates.- 'constant' is a constant learning rate given by 'learning_rate_init'.- 'invscaling' gradually decreases the learning rate at each time step 't' using an inverse scaling exponent of 'power_t'. effective_learning_rate = learning_rate_init / pow(t, power_t)- 'adaptive' keeps the learning rate constant to 'learning_rate_init' as long as training loss keeps decreasing. Each time two consecutive epochs fail to decrease training loss by at least tol, or fail to increase validation score by at least tol if 'early_stopping' is on, the current learning rate is divided by 5.Only used when ``solver='sgd'``.",'constant'
,"learning_rate_init learning_rate_init: float, default=0.001The initial learning rate used. It controls the step-sizein updating the weights. Only used when solver='sgd' or 'adam'.",0.001
,"power_t power_t: float, default=0.5The exponent for inverse scaling learning rate.It is used in updating effective learning rate when the learning_rateis set to 'invscaling'. Only used when solver='sgd'.",0.5
,"max_iter max_iter: int, default=200Maximum number of iterations. The solver iterates until convergence(determined by 'tol') or this number of iterations. For stochasticsolvers ('sgd', 'adam'), note that this determines the number of epochs(how many times each data point will be used), not the number ofgradient steps.",50
,"shuffle shuffle: bool, default=TrueWhether to shuffle samples in each iteration. Only used whensolver='sgd' or 'adam'.",True
,"random_state random_state: int, RandomState instance, default=NoneDetermines random number generation for weights and biasinitialization, train-test split if early stopping is used, and batchsampling when solver='sgd' or 'adam'.Pass an int for reproducible results across multiple function calls.See :term:`Glossary `.",1


In [4]:
pred = clf.predict(hog_train)
train_accuracy = np.mean(pred == labels_train)
print(train_accuracy)

1.0


In [5]:
pred_test = clf.predict(hog_test)

## Deep network fun

In [6]:

import tensorflow as tf
import keras
from keras.models import Sequential
from keras.layers import Dense, Conv1D, Flatten, Activation
from keras import regularizers
from sklearn.model_selection import train_test_split

from keras.callbacks import EarlyStopping
from keras.callbacks import CSVLogger
from keras.callbacks import ProgbarLogger



In [7]:





# ----------- Dataset class -----------
#This will only store the data we will need in a data object


class DataGenerator:
    def __init__(self, images, transforms,labels, batch_size=8, shuffle=True):
        self.images = images #The image
        self.transforms = transforms #The feature representation of the image (HOG or SIFT)
        self.labels = labels
        #self.batch_size = batch_size
        self.shuffle = shuffle
        #self.indices = np.arange(len(self.images))

    def __len__(self):
        #return int(np.ceil(len(self.images) / self.batch_size))
        return len(self.images)
    def __getitem__(self, idx):
        #batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]

        #batch_image = np.array([self.images[i] for i in batch_idx])
        #batch_image2 = np.expand_dims(batch_image, axis=0)

        #batch_hog = np.array([self.transforms[i] for i in batch_idx])
        #batch_hog2 = np.expand_dims(batch_hog, axis=0)

        
        #batch_label = np.array([self.labels[i] for i in batch_idx])
        
        image = self.images[idx]
        #image = np.expand_dims(image, axis=0)
        hog = self.transforms[idx]
        #hog = np.expand_dims(hog, axis=0)
        label = self.labels[idx]
        
        
        return (image, hog), (label)
    def __call__(self):
        for idx in range(self.__len__()):
            data = self.__getitem__(idx)
            yield data



    #idk if I need a yield here (call)





In [8]:
def get_backbone(model_name, input_shape=(128,128,3)):
    if model_name == "resnet50":
        base = tf.keras.applications.ResNet50(
            weights="imagenet",
            include_top=False,
            input_shape=input_shape
        )
    elif model_name == "efficientnet_b0":
        base = tf.keras.applications.EfficientNetB0(
            weights="imagenet",
            include_top=False,
            input_shape=input_shape
        )
    elif model_name == "efficientnet_b4":
        base = tf.keras.applications.EfficientNetB4(
            weights="imagenet",
            include_top=False,
            input_shape=input_shape
        )
    else:
        raise ValueError(f"Unknown model {model_name}")

    return base

In [9]:
def HOG_model(image_size,transform_size,backbone_name="resnet50"):
    
    ##############################
    #Bottom branch implementation
    ##############################
    input_HOG = tf.keras.layers.Input(shape=(972,1),name="HOG_features")
    #weird flatten?
    x = tf.keras.layers.Flatten()(input_HOG)
    #Normalisation
    x = tf.keras.layers.BatchNormalization()(x)   
    #Pass this data to a feedforward neurlal network
    x = tf.keras.layers.Dense(256, activation='relu')(x)
    x = tf.keras.layers.Dense(128, activation='relu')(x)
    x = tf.keras.layers.Dense(64, activation='relu')(x)

    ##############################
    #Top branch implementation
    ##############################
    input_image = tf.keras.layers.Input(shape=(128,128,3),name="face_images")
    #Normalisation
    #input_image_BN = tf.keras.layers.BatchNormalization()(input_image) #Article uses raw images  
    #Pass through a pretrained network
    base_model = get_backbone(backbone_name)

    # Optional: freeze backbone
    base_model.trainable = False
    x2 = base_model(input_image, training=False)
    #This is passed to a 64 channel 3x3  2D convolutional filter
    x2 = tf.keras.layers.Conv2D(filters=16, kernel_size=(3,3), activation='relu', padding='same')(x2)
    # Max pooling layer with 2x2 window
    x2 = tf.keras.layers.MaxPooling2D(pool_size=(2,2))(x2)
    # Flatten the output
    x2 = tf.keras.layers.Flatten()(x2)
    # Dense layer with 64 ReLU-activated units
    image_embedding = tf.keras.layers.Dense(64, activation='relu')(x2)

    ##############################
    #Concatenation of branches implementation
    ##############################
    # Concatenate image and HOG features
    concat_features = tf.keras.layers.Concatenate()([image_embedding, x])
    #One more fully connected block
    x3 = tf.keras.layers.Dense(64, activation='relu')(concat_features)
    x3 = tf.keras.layers.Dense(32, activation='relu')(x3)
    x3 = tf.keras.layers.Dropout(0.4)(x3)

    # Output layer: adjust units and activation for your task
    num_classes = 3  # example for classification
    output = tf.keras.layers.Dense(num_classes, activation='softmax')(x3)

    model = tf.keras.Model(inputs=[input_image, input_HOG], outputs=output)

    return model

In [10]:
#Initialise model and show summary
#model.summary()

# Training the model

In [11]:
#Build data containers and get train, test, validation split, also configure model parameters here
#Generate indices to create the train-validation split
n = 80
batch_size = 8
base_indices = np.arange(n)
train_base, val_base = train_test_split(
    base_indices,
    test_size=0.2,
    random_state=42,
    shuffle=True
)
train_indices = np.concatenate([train_base, train_base + n])
val_indices   = np.concatenate([val_base, val_base + n])

#feature split
features_hog_train = hog_train[train_indices]      
features_hog_val   = hog_train[val_indices]

features_sift_train = sift_train[train_indices]      
features_sift_val   = sift_train[val_indices]


#label split
y_train = labels_train[train_indices]
y_val   = labels_train[val_indices]
#faces split
faces_train = images_train2[train_indices]
faces_val   = images_train2[val_indices]

#Create data generators
train_generator = DataGenerator(faces_train,features_hog_train,y_train,batch_size=batch_size)
val_generator   = DataGenerator(faces_val,features_hog_val,y_val,batch_size=batch_size)

#Shape of the output of the data generator
output_signature = (
        (
            tf.TensorSpec(shape=(128, 128,3), dtype=tf.float32),  # images
            tf.TensorSpec(shape=(972,), dtype=tf.float32),         # HOG feature
                   
        ),
        tf.TensorSpec(shape=(), dtype=tf.int8),  # Label
    )
#Finalise the datasets
#training dataset
train_dataset = tf.data.Dataset.from_generator(
        train_generator,
        output_signature=output_signature
    )
train_dataset = train_dataset.batch(batch_size)  
train_dataset = train_dataset.prefetch(tf.data.AUTOTUNE)
#validation dataset
val_dataset =  tf.data.Dataset.from_generator(
        val_generator,
        output_signature=output_signature
    )
val_dataset = val_dataset.batch(batch_size)  
val_dataset = val_dataset.prefetch(tf.data.AUTOTUNE)






In [12]:
model = HOG_model((128,128,3),(972,1),"resnet50")
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

csv_logger = CSVLogger("training_log.csv")

early_stopping = EarlyStopping(
    monitor='val_loss',  # Metric to monitor (validation loss)
    patience=10,         # Number of epochs with no improvement before stopping
    restore_best_weights=True  # Restores model weights from epoch with best value
)


model.fit(train_dataset,
            epochs=20,
            batch_size=batch_size,
            validation_data=val_dataset,
            verbose = 1,
            callbacks = [csv_logger,ProgbarLogger(),early_stopping])

#Make a prediction for the test_data somehow

probs = model.predict([images_test, hog_test])
pred_test = np.argmax(probs, axis=1)



Epoch 1/20
     16/Unknown 7s 96ms/step - accuracy: 0.3745 - loss: 1.4311

c:\Users\bramo\Documents\Unief5\Semester2\ComputerVision\ComputerVision\venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


16/16 ━━━━━━━━━━━━━━━━━━━━ 9s 228ms/step - accuracy: 0.3828 - loss: 1.3843 - val_accuracy: 0.6250 - val_loss: 0.8640
Epoch 2/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 117ms/step - accuracy: 0.6562 - loss: 0.7840 - val_accuracy: 0.7188 - val_loss: 0.6100
Epoch 3/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 117ms/step - accuracy: 0.8047 - loss: 0.4272 - val_accuracy: 0.6562 - val_loss: 0.8501
Epoch 4/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 129ms/step - accuracy: 0.9531 - loss: 0.1435 - val_accuracy: 0.6562 - val_loss: 0.8435
Epoch 5/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 138ms/step - accuracy: 0.9688 - loss: 0.0885 - val_accuracy: 0.6250 - val_loss: 1.3020
Epoch 6/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 149ms/step - accuracy: 0.9766 - loss: 0.0593 - val_accuracy: 0.6250 - val_loss: 2.1111
Epoch 7/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 134ms/step - accuracy: 1.0000 - loss: 0.0204 - val_accuracy: 0.7500 - val_loss: 1.1225
Epoch 8/20
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 134ms/step - accuracy: 1.0000 - loss: 0.0124 - val_accuracy: 0.7500 - val_

In [13]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

#SVM on hog

# Scale features (important for SVMs)
scaler = StandardScaler()
hog_train_scaled = scaler.fit_transform(hog_train)
hog_test_scaled  = scaler.transform(hog_test)

# Train SVM
svm = SVC(kernel='rbf', C=10, gamma='scale', decision_function_shape='ovr')
svm.fit(hog_train_scaled, labels_train.squeeze())

# Evaluate on training set
train_acc = np.mean(svm.predict(hog_train_scaled) == labels_train.squeeze())
print(f"Train accuracy: {train_acc:.4f}")

# Predict on test set
pred_test = svm.predict(hog_test_scaled)

Train accuracy: 1.0000


In [14]:
#SVM on hog and sift
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Concatenate HOG and SIFT into one feature vector per sample
combined_train = np.concatenate([features_hog_train, features_sift_train], axis=1)
combined_test  = np.concatenate([hog_test,  sift_test],  axis=1)
combined_val = np.concatenate([features_hog_val, features_sift_val], axis=1)


# SVM pipeline with scaling
svm = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', SVC(kernel='rbf', C=10, gamma='scale', 
                decision_function_shape='ovr', probability=True))
])

svm.fit(combined_train, y_train.squeeze())

train_acc = np.mean(svm.predict(combined_train) == y_train.squeeze())
print(f"Train accuracy: {train_acc:.4f}")

# Evaluate on validation set
val_preds = svm.predict(combined_val)
val_acc   = np.mean(val_preds == y_val.squeeze())
print(f"Validation accuracy: {val_acc:.4f}")

pred_test = svm.predict(combined_test)

Train accuracy: 1.0000
Validation accuracy: 0.7188


In [15]:
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV

# Concatenate HOG and SIFT into one feature vector per sample
combined_train = np.concatenate([features_hog_train, features_sift_train], axis=1)
combined_test  = np.concatenate([hog_test,  sift_test],  axis=1)
combined_val = np.concatenate([features_hog_val, features_sift_val], axis=1)

pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    SVC(decision_function_shape='ovr'))
])

param_grid = {
    'clf__kernel': ['rbf', 'linear'],
    'clf__C':      [0.01, 0.1, 1, 10],
    'clf__gamma':  ['scale', 'auto', 0.001, 0.01]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='accuracy', refit=True)
grid_search.fit(combined_train, y_train.squeeze())

# Print accuracy for every combination
print(f"{'Kernel':<10} {'C':<8} {'Gamma':<10} {'CV Acc':<12} {'Train Acc':<12} {'Val Acc'}")
print("-" * 65)

results = grid_search.cv_results_
for i in range(len(results['params'])):
    params     = results['params'][i]
    cv_acc     = results['mean_test_score'][i]




    # Strip the 'clf__' prefix from param keys
    raw_params = {k.replace('clf__', ''): v for k, v in params.items()}

    temp_svm = Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    SVC(decision_function_shape='ovr', **raw_params))
    ])

    temp_svm.fit(combined_train, y_train.squeeze())
    train_acc = np.mean(temp_svm.predict(combined_train) == y_train.squeeze())
    val_acc   = np.mean(temp_svm.predict(combined_val)   == y_val.squeeze())

    print(f"{params['clf__kernel']:<10} {params['clf__C']:<8} {str(params['clf__gamma']):<10} "
          f"{cv_acc:<12.4f} {train_acc:<12.4f} {val_acc:.4f}")

print("-" * 65)
print(f"Best params:    {grid_search.best_params_}")
print(f"Best CV acc:    {grid_search.best_score_:.4f}")

# Final evaluation with best model
best_svm   = grid_search.best_estimator_
val_acc    = np.mean(best_svm.predict(combined_val)   == y_val.squeeze())
train_acc  = np.mean(best_svm.predict(combined_train) == y_train.squeeze())
print(f"Best train acc: {train_acc:.4f}")
print(f"Best val acc:   {val_acc:.4f}")

pred_test = best_svm.predict(combined_test)

Kernel     C        Gamma      CV Acc       Train Acc    Val Acc
-----------------------------------------------------------------
rbf        0.01     scale      0.3754       0.3750       0.3750
linear     0.01     scale      0.8668       1.0000       0.8125
rbf        0.01     auto       0.3754       0.3750       0.3750
linear     0.01     auto       0.8668       1.0000       0.8125
rbf        0.01     0.001      0.3754       0.3750       0.3750
linear     0.01     0.001      0.8668       1.0000       0.8125
rbf        0.01     0.01       0.3754       0.3750       0.3750
linear     0.01     0.01       0.8668       1.0000       0.8125
rbf        0.1      scale      0.3754       0.3750       0.3750
linear     0.1      scale      0.8668       1.0000       0.8125
rbf        0.1      auto       0.3754       0.3750       0.3750
linear     0.1      auto       0.8668       1.0000       0.8125
rbf        0.1      0.001      0.3754       0.3750       0.3750
linear     0.1      0.001      0.8668

In [22]:
# Stage 1: is this a lookalike (class 3) or a real person (class 1 or 2)?
# Stage 2: if real person, is it person 1 or person 2?

# Binary label: 0 = real person, 1 = lookalike
lookalike_labels_train = (y_train.squeeze() == 2).astype(int)

svm_stage1 = Pipeline([('scaler', StandardScaler()), ('clf', SVC(kernel='rbf', C=1))])
svm_stage1.fit(features_sift_train, lookalike_labels_train)

# Only train stage 2 on real persons (classes 0 and 1)
real_mask  = y_train.squeeze() != 2
svm_stage2 = Pipeline([('scaler', StandardScaler()), ('clf', SVC(kernel='rbf', C=1))])
svm_stage2.fit(features_sift_train[real_mask], y_train.squeeze()[real_mask])

# Inference
def predict_hierarchical(X):
    is_lookalike = svm_stage1.predict(X)
    final_preds  = np.where(is_lookalike, 2, svm_stage2.predict(X))
    return final_preds

val_preds = predict_hierarchical(features_sift_val)
print(f"Hierarchical val accuracy: {np.mean(val_preds == y_val.squeeze()):.4f}")

pred_test = predict_hierarchical(sift_test)

Hierarchical val accuracy: 0.9375


In [23]:
# 1. Generate final predictions using the pipeline
# This automatically extracts features from test_X and classifies them
test_y_star = pred_test

# 2. Prepare the submission dataframe
# Assuming 'test' is the original dataframe containing image info
submission = test.copy().drop('img', axis=1)
submission['class'] = test_y_star

# 3. Save to CSV for Kaggle upload
submission.to_csv('submission.csv', index=True)
print("✅ Final submission.csv created!")
submission.head()

✅ Final submission.csv created!


,class
id,
0,1
1,0
2,0
3,1
4,1
